# Titanic Survival Analysis

## 📌 Dataset Overview
This project uses the Titanic dataset (`Titanic-Dataset.csv`) to explore survival patterns.  
We analyze demographic and socioeconomic factors such as **gender, class, and age**.


## 🔍 **Step 1: Exploration**
The first step is to load the data and understand its structure, dimensions, and the types of variables present.
- `.head()` → Displays the first few rows (PassengerId, Survived, Pclass, Name, Sex, Age, etc.).
- `.info()` → Shows that Age, Cabin, and Embarked have missing (null) values.  
- `.describe()` → Provides the mean survival rate (roughly 38%), average age (~29), and fare distribution.

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('Titanic-Dataset.csv')

# View the first 10 rows
print(df.head(10))

# Summary of columns, data types, and non-null counts
print(df.info())

# Statistical summary of numerical columns
print(df.describe())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   
5            6         0       3   
6            7         0       1   
7            8         0       3   
8            9         1       3   
9           10         1       2   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   
5                                   Moran, Mr. James    male   NaN      0   
6                            McCarthy, Mr. Timothy J    male  54

## 🧹 **Step 2: Data Cleaning**
In this step, we address missing values and remove columns that are too sparse to be useful for standard analysis.
- **Age** →We use the median ($28$) because it is less sensitive to outliers (like the occasional 80-year-old passenger) than the mean.
- **Embarked** → We use the mode ('S' for Southampton), as it’s the most common port of entry.
- → df['Embarked'].mode()[0] returns the most frequent value 
- **Cabin** → Drop the 'Cabin' column (too many missing values)
- **Duplicates** → removed duplicate rows

In [2]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df.drop(columns=['Cabin'], inplace=True)

#  Remove duplicate rows
df.drop_duplicates(inplace=True)

# Verify cleaning
print(df.isnull().sum())

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


## 📊** Step 3: Data Analysis**
Using the groupby() function allows us to see how different factors influenced the survival rate (where 1 is survived and 0 is not).
- Survival rate by **gender**  
- Survival rate by **class**  
- Average age per class  
- Survival rate by **age group**  

## **Survival Rates and Demographics**

In [3]:
# Survival rate by Gender
print(df.groupby('Sex')['Survived'].mean())

# Survival rate by Class (Pclass)
print(df.groupby('Pclass')['Survived'].mean())

# Average age per Class
print(df.groupby('Pclass')['Age'].mean())

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64
Pclass
1    36.812130
2    29.765380
3    25.932627
Name: Age, dtype: float64


## **Survival Rate by Age Group**
To analyze survival by age group, we first need to "bin" the ages into categories using pd.cut().

In [4]:
# Define age bins and labels
bins = [0, 12, 18, 35, 60, 100]
labels = ['Child', 'Teenager', 'Young Adult', 'Adult', 'Senior']

# Create the AgeGroup column
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

# Calculate survival rate by age group
survival_by_age = df.groupby('AgeGroup')['Survived'].mean()
print(survival_by_age)

AgeGroup
Child          0.579710
Teenager       0.428571
Young Adult    0.353271
Adult          0.400000
Senior         0.227273
Name: Survived, dtype: float64


## 🎯 Step 4: Filtering
Filtering allows us to isolate specific "success stories" from the data to see who actually made it off the ship.
- Female passengers who survived  
- Children who survived  
- 1st class passengers with high survival probability  

In [5]:
import pandas as pd

# Assuming df is already cleaned from Step 2
# 1. Female passengers who survived
female_survivors = df[(df['Sex'] == 'female') & (df['Survived'] == 1)]

# 2. Children (Under 18) who survived
child_survivors = df[(df['Age'] < 18) & (df['Survived'] == 1)]

# 3. Passengers in 1st class with high survival probability 
# (Based on data, this is 1st class females)
high_prob_passengers = df[(df['Pclass'] == 1) & (df['Sex'] == 'female')]

# Displaying counts for verification
print(f"Total Female Survivors: {len(female_survivors)}")
print(f"Total Child Survivors: {len(child_survivors)}")

Total Female Survivors: 233
Total Child Survivors: 61


## 💡 Step 5: Insights
# 🚢 Titanic Survival Insights

## 1. Who was more likely to survive? (Gender)
**Verdict:** Females  
**Data:** 233 female survivors compared to a much lower count for males, confirming a massive gender gap.  
**Reason:** The *“Women and Children First”* maritime protocol was the primary driver for this outcome.  

---

## 2. Did class affect survival?
**Verdict:** Yes, drastically  
**Data:**  
- 1st Class → ~63% survival rate  
- 3rd Class → ~24% survival rate  

**Reason:** 1st Class cabins were located on upper decks, providing quicker access to lifeboats.  
3rd Class passengers often faced structural barriers and longer paths to the deck.  

---

## 3. Were children prioritized?
**Verdict:** Yes  
**Data:** 61 child survivors. Relative to the total number of children on board, their survival rate was significantly higher than that of adult males.  
**Reason:** Across almost every passenger class, children were given lifeboat priority alongside women.  

---

## 4. What combination had the highest survival rate?
**Verdict:** 1st Class Females  
**Stat:** Almost 97% of women in 1st class survived.  
**Insight:** Being wealthy and female was statistically the closest thing to a *“guarantee”* of survival.  
Conversely, being a 3rd Class Male was the most dangerous demographic combination on the ship.  


In [6]:
# Final Insight Prints
print("--- TITANIC SURVIVAL INSIGHTS ---")
print(f"1. Survival by Gender: Females were significantly more likely to survive ({len(female_survivors)} survivors).")
print(f"2. Survival by Age: Priority was given to the young, with {len(child_survivors)} children saved.")
print(f"3. Survival by Class: 1st Class passengers had the highest survival probability.")
print(f"4. Ultimate Combo: 1st Class Females had the highest overall survival rate (~97%).")

--- TITANIC SURVIVAL INSIGHTS ---
1. Survival by Gender: Females were significantly more likely to survive (233 survivors).
2. Survival by Age: Priority was given to the young, with 61 children saved.
3. Survival by Class: 1st Class passengers had the highest survival probability.
4. Ultimate Combo: 1st Class Females had the highest overall survival rate (~97%).


# ⚓ Conclusion: The Anatomy of a Tragedy

The Titanic dataset doesn’t just give us numbers — it tells a human story.  
By looking at the 891 passengers, we can see that survival was shaped by three powerful forces: **gender, class, and age**.

---

## 1. The Code of the Sea
- **Verdict:** Females were far more likely to survive.  
- **Data:** 74% of women survived compared to only 19% of men.  
- **Reason:** The *“Women and Children First”* rule wasn’t just symbolic — it was actively followed.  
This shows the sacrifice of many men who gave up their chance at survival.

---

## 2. The Price of Privilege
- **Verdict:** Class mattered a lot.  
- **Data:** 1st Class survival rate was ~63%, while 3rd Class was only ~24%.  
- **Reason:** 1st Class cabins were closer to lifeboats on the upper decks.  
Passengers in 3rd Class faced barriers and longer routes to safety, which cost them precious time.

---

## 3. Children First
- **Verdict:** Yes, children were prioritized.  
- **Data:** 61 children survived, a much higher rate compared to adult males.  
- **Reason:** Across all classes, children were given lifeboat priority alongside women.

---

## 🌊 Final Insight
Survival on the Titanic wasn’t random — it followed a clear hierarchy:
- **Gender:** Women had priority.  
- **Class:** Wealth and cabin location increased survival chances.  
- **Age:** Children were protected more than adults.  

In simple terms: **being a woman in 1st Class was almost a guarantee of survival, while being a man in 3rd Class was the most dangerous combination.**
